<a href="https://colab.research.google.com/github/Yae-sine/Language-Model-Inversion/blob/main/llama2/llama2_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Section 1 — Testing & Understanding**

### Prerequisites — Running This Notebook

Before running the notebook in Google Colab:

1. **Enable GPU:** Go to `Runtime → Change runtime type → T4 GPU`.
2. **Install dependencies:** Run the following cell:
   ```python
   %pip install -q -U transformers accelerate bitsandbytes datasets huggingface_hub
   ```
3. **Hugging Face access:** Request access to `meta-llama/Llama-2-7b-hf` on Hugging Face.
4. **Authenticate:** Run:
   ```python
   from huggingface_hub import notebook_login
   notebook_login()
   ```
   Enter your Hugging Face access token. **Never share or commit your token to GitHub.**
5. **Execute the remaining cells in order** to load Llama 2, load the dataset, and run the extraction tests.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset


In [ ]:
model_name = "meta-llama/Llama-2-7b-hf"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model.eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print(model.device)
print(torch.cuda.memory_allocated() / 1024**3, "GB")

cuda:0
8.55749797821045 GB


In [ ]:
instruction = "What is the capital of France?"

inputs = tokenizer(instruction, return_tensors="pt").to("cuda")

# i added this to avoid training llama

with torch.no_grad():
    outputs = model(**inputs)

#to select the final token position
logits = outputs.logits[:, -1, :]

print("Input tokens:", inputs.input_ids.shape)
print(logits)
print("Logits shape:", logits.shape)

Input tokens: torch.Size([1, 8])
tensor([[-5.8555, -5.1055, 11.8203,  ..., -2.5449, -3.7344, -1.1211]],
       device='cuda:0', dtype=torch.float16)
Logits shape: torch.Size([1, 32000])


In [ ]:
log_probs = torch.log_softmax(logits.float(), dim=-1)

print("Vector shape:", log_probs.shape)
print("First 10 values:", log_probs[0, :10])
print("Sum of probabilities:", log_probs.exp().sum().item())

Vector shape: torch.Size([1, 32000])
First 10 values: tensor([-22.1368, -21.3868,  -4.4610, -13.9044, -14.7765, -15.5231, -14.9874,
        -17.1100, -17.6964, -14.8116], device='cuda:0')
Sum of probabilities: 1.0000005960464478


In [ ]:
#i converted it into proper ordinary probabilities
#from log to expo
probs = log_probs.exp()

values, indices = torch.topk(probs[0], 5)

for token_id, probability in zip(indices, values):
    token = tokenizer.decode([token_id.item()])
    print(repr(token), ":", round(probability.item(), 4))

'\n' : 0.6661
'Paris' : 0.0605
'What' : 0.0295
'The' : 0.0214
'(' : 0.0132


In [ ]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

instruction = dataset[0]["instruction"]

inputs = tokenizer(instruction, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[:, -1, :]
log_probs = torch.log_softmax(logits.float(), dim=-1)

print("Instruction:", instruction)
print("Vector shape:", log_probs.shape)

Instruction: When did Virgin Australia start operating?
Vector shape: torch.Size([1, 32000])


In [ ]:
instructions = dataset[:10]["instruction"]

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

inputs = tokenizer(
    instructions,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512
).to("cuda")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[:, -1, :]
log_probs = torch.log_softmax(logits.float(), dim=-1)

print("Number of instructions:", len(instructions))
print("Vectors shape:", log_probs.shape)

Number of instructions: 10
Vectors shape: torch.Size([10, 32000])


## **Section 2 — Complete Extraction Pipeline**

In [ ]:
import numpy as np

batch_size = 4
vectors = []
instructions = dataset["instruction"]

model.eval()

for start in range(0, len(instructions), batch_size):
    batch = instructions[start:start + batch_size]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, -1, :]
    log_probs = torch.log_softmax(logits.float(), dim=-1)

    vectors.append(log_probs.cpu().numpy())

    if start % 100 == 0:
        print(f"Processed {start}/{len(instructions)}")

vectors = np.concatenate(vectors, axis=0)

print("Final shape:", vectors.shape)

Processed 0/15011
Processed 100/15011
Processed 200/15011
Processed 300/15011
Processed 400/15011
Processed 500/15011
Processed 600/15011
Processed 700/15011
Processed 800/15011
Processed 900/15011
Processed 1000/15011
Processed 1100/15011
Processed 1200/15011
Processed 1300/15011
Processed 1400/15011
Processed 1500/15011
Processed 1600/15011
Processed 1700/15011
Processed 1800/15011
Processed 1900/15011
Processed 2000/15011
Processed 2100/15011
Processed 2200/15011
Processed 2300/15011
Processed 2400/15011
Processed 2500/15011
Processed 2600/15011
Processed 2700/15011
Processed 2800/15011
Processed 2900/15011
Processed 3000/15011
Processed 3100/15011
Processed 3200/15011
Processed 3300/15011
Processed 3400/15011
Processed 3500/15011
Processed 3600/15011
Processed 3700/15011
Processed 3800/15011
Processed 3900/15011
Processed 4000/15011
Processed 4100/15011
Processed 4200/15011
Processed 4300/15011
Processed 4400/15011
Processed 4500/15011
Processed 4600/15011
Processed 4700/15011
Proc

In [ ]:
import os

output_file = os.path.join(output_dir, "llama2_dolly15k.npz")

np.savez_compressed(
    output_file,
    instructions=np.array(instructions),
    vectors=vectors
)

print("Saved:", output_file)
print("File size:", round(os.path.getsize(output_file) / (1024**2), 2), "MB")

Saved: /content/llama2_results/llama2_dolly15k.npz
File size: 1115.07 MB


In [32]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [33]:
import shutil

destination = "/content/drive/MyDrive/llama2_dolly15k.npz"

shutil.copy2(output_file, destination)

print("Saved to Google Drive:", destination)

Saved to Google Drive: /content/drive/MyDrive/llama2_dolly15k.npz
